In [ ]:
# Pin the branch so a later push cannot silently change what a finished run
# was produced by. If the repo is private, add a token:
#   https://<PAT>@github.com/Rifat137710/Public.git
!rm -rf /kaggle/working/safesac-repo
!git clone -q --branch claude/thesis-q1-journal-path-komv2c \
    https://github.com/Rifat137710/Public.git /kaggle/working/safesac-repo
%cd /kaggle/working/safesac-repo
!git rev-parse --short HEAD


In [ ]:
!pip -q install -r requirements.txt 2>&1 | tail -3
import torch, pandapower, cvxpy, clarabel, gymnasium
print("torch", torch.__version__)
print("pandapower", pandapower.__version__, "(must be 3.2.0)")
print("cvxpy", cvxpy.__version__, "| gymnasium", gymnasium.__version__)


In [ ]:
!python -m pytest tests/test_powerflow.py -q 2>&1 | tail -4
!python -u scripts/learned_source.py --seeds 0 --episodes 3 --eval-episodes 2 \
    --eval-z 6.0 --refresh 1 288 --out-dir /kaggle/working/smoke \
    2>&1 | grep -v "UserWarning\|warnings.warn" | tail -12


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

!python -u scripts/learned_source.py \
    --seeds 0 1 2 \
    --episodes 200 \
    --eval-episodes 20 \
    --alpha 0.003 \
    --train-z 6.0 \
    --eval-z 6.0 8.0 \
    --refresh 1 12 48 288 \
    --load 0.40 \
    --evs 30 \
    --out-dir /kaggle/working/results/learned_source \
    2>&1 | grep -v "UserWarning\|warnings.warn"


In [ ]:
import json, shutil
from pathlib import Path

d = json.loads(
    Path("/kaggle/working/results/learned_source/learned_source.json").read_text())
print("train Z", d["train_z_pct"], "| eval Z", d["eval_z_pct"],
      "| refresh", d["refresh"], "| seeds", d["seeds"],
      "| fingerprint", d["fingerprint"])
cols = ["raw"] + [str(r) for r in d["refresh"]]
print(f"{'Z%':>6}" + "".join(f"{c:>14}" for c in cols))
for z, row in d["summary"].items():
    print(f"{z:>6}" + "".join(f"{row[c]['viol'][0]:>14.4f}" for c in cols))

shutil.make_archive("/kaggle/working/learned_source_results", "zip",
                    "/kaggle/working/results")
print("\ndownload /kaggle/working/learned_source_results.zip and send it back")
